In [ ]:
import os
os.environ.setdefault("DATASET_PATH", "/workspace/data")     # for ETH datamodule
os.environ.setdefault("ETH_SUBPATH",  "eth_dataset")         # for ETH datamodule
os.environ.setdefault("DATA_ROOT",    "/workspace/base/src/data")  # shared root (optional)


In [ ]:
# Hand (left/right) mirroring augmentation — Jupyter cell version

import os, re, json, shutil, random
from pathlib import Path
from typing import List, Tuple, Optional, Dict

import numpy as np
import pandas as pd

# ---------- ID helpers ----------
ID_RX = re.compile(r"^id_(\d+)$")

def iter_numeric_id_dirs(root: Path) -> List[Path]:
    pairs = []
    for p in root.glob("id_*"):
        if p.is_dir():
            m = ID_RX.match(p.name)
            if m: pairs.append((int(m.group(1)), p))
    pairs.sort(key=lambda t: t[0])
    return [p for _, p in pairs]

def next_free_id(root: Path, start_from: Optional[int] = None) -> int:
    existing = {int(ID_RX.match(p.name).group(1)) for p in iter_numeric_id_dirs(root)}
    baseline = (max(existing) + 1) if existing else 1
    i = max(baseline, start_from or baseline)
    while i in existing:
        i += 1
    return i

# ---------- JSON helpers ----------
def load_json_safe(p: Path) -> Optional[dict]:
    if not p.exists(): return None
    try:
        with open(p, "r") as f:
            return json.load(f)
    except Exception:
        return None

def save_json_safe(p: Path, obj: dict):
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w") as f:
        json.dump(obj, f, indent=2)

# ---------- handedness discovery ----------
def discover_handedness(pid_dir: Path) -> Optional[str]:
    for posture in ("sitting", "standing"):
        fb = pid_dir / posture / "feedback.json"
        data = load_json_safe(fb)
        if data and isinstance(data, dict):
            val = data.get("handedness", None)
            if isinstance(val, str) and val.lower() in ("left", "right"):
                return val.lower()
    return None

# ---------- column resolution ----------
def resolve_col(df: pd.DataFrame, wanted: str) -> Optional[str]:
    if wanted in df.columns: return wanted
    alias = {
        "Acc_X":"AccX","Acc_Y":"AccY","Acc_Z":"AccZ",
        "Gyro_X":"GyroX","Gyro_Y":"GyroY","Gyro_Z":"GyroZ",
        "AccX":"AccX","AccY":"AccY","AccZ":"AccZ",
        "GyroX":"GyroX","GyroY":"GyroY","GyroZ":"GyroZ",
    }.get(wanted, None)
    return alias if (alias in df.columns) else None

def triple_cols(df: pd.DataFrame, prefix: str) -> Tuple[Optional[str], Optional[str], Optional[str]]:
    X = resolve_col(df, f"{prefix}_X") or resolve_col(df, f"{prefix}X")
    Y = resolve_col(df, f"{prefix}_Y") or resolve_col(df, f"{prefix}Y")
    Z = resolve_col(df, f"{prefix}_Z") or resolve_col(df, f"{prefix}Z")
    return X, Y, Z

def list_ppg_columns(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if c.startswith("PPG")]

# ---------- engine ----------
class HandAugmentationEngine:
    """
    Create mirrored participants (opposite handedness) as new id_* folders.
    - IMU mirror: AccX → -AccX; GyroX → -GyroX; GyroZ → -GyroZ (sagittal reflection)
    - PPG: swap symmetric PD pairs when both exist (1↔2, 3↔4, …)
    - feedback.json: flipped handedness + augmented_from + updated id
    """

    def __init__(self, data_dir: Path, seed: int = 42):
        self.data_dir = Path(data_dir)
        random.seed(seed); np.random.seed(seed)

    # ---- IMU + PPG mirroring ----
    def mirror_sensor_data(self, df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()

        # Acc
        Ax, Ay, Az = triple_cols(out, "Acc")
        if Ax: out[Ax] = -out[Ax]

        # Gyro
        Gx, Gy, Gz = triple_cols(out, "Gyro")
        if Gx: out[Gx] = -out[Gx]
        if Gz: out[Gz] = -out[Gz]

        # PPG: swap (1↔2), (3↔4), … up to 19↔20 if present
        def ppg_name(i: int) -> Optional[str]:
            if f"PPG{i}" in out.columns: return f"PPG{i}"
            if f"PPG_{i}" in out.columns: return f"PPG_{i}"
            return None
        for i in range(1, 20, 2):
            a, b = ppg_name(i), ppg_name(i+1)
            if a and b:
                tmp = out[a].copy(); out[a] = out[b]; out[b] = tmp

        return out

    def update_feedback(self, src_feedback: Optional[dict], new_id_num: int, original_handedness: Optional[str]) -> dict:
        out = dict(src_feedback or {})
        out["id"] = str(new_id_num)
    
        # Deterministic flip:
        if original_handedness == "left":
            out["handedness"] = "right"
            out["handedness_source"] = "measured"
        elif original_handedness == "right":
            out["handedness"] = "left"
            out["handedness_source"] = "measured"
        else:
            # If original unknown, keep unknown (or set to "left"/"right" based on your assumption outside)
            out["handedness"] = "unknown"
            out["handedness_source"] = "unknown"
    
        out["augmented_from"] = (src_feedback or {}).get("id", "unknown")
        out["augmentation_type"] = "hand_mirroring"
        return out


    # ---- copying / writing ----
    def create_mirrored_participant(self, src_dir: Path, new_id_num: int, dry_run: bool) -> bool:
        tgt_dir = self.data_dir / f"id_{new_id_num}"
        print(f"  → {src_dir.name}  ⇢  {tgt_dir.name}")

        if dry_run:
            total = 0
            for posture_dir in [p for p in src_dir.iterdir() if p.is_dir()]:
                n = len(list(posture_dir.glob("*.csv"))); total += n
                print(f"     (dry-run) would write {n:4d} files in {tgt_dir.name}/{posture_dir.name}/")
            print(f"     (dry-run) total files: {total}")
            return True

        try:
            tgt_dir.mkdir(parents=True, exist_ok=True)
            files_processed = 0
            handed = discover_handedness(src_dir) or "unknown"

            for posture_dir in [p for p in src_dir.iterdir() if p.is_dir()]:
                tgt_posture = tgt_dir / posture_dir.name
                tgt_posture.mkdir(exist_ok=True)

                # CSVs
                for csv in posture_dir.glob("*.csv"):
                    try:
                        df = pd.read_csv(csv)
                        mirrored = self.mirror_sensor_data(df)
                        mirrored.to_csv(tgt_posture / csv.name, index=False)
                        files_processed += 1
                    except Exception as e:
                        print(f"     {csv.name}: {e}")

                # feedback.json
                src_fb = load_json_safe(posture_dir / "feedback.json")
                fb_out = self.update_feedback(src_fb, new_id_num, handed)
                save_json_safe(tgt_posture / "feedback.json", fb_out)

            print(f"     wrote {files_processed} CSV files")
            return True

        except Exception as e:
            print(f"     failed to create {tgt_dir.name}: {e}")
            if tgt_dir.exists():
                shutil.rmtree(tgt_dir)
            return False

    # ---- scanning & orchestration ----
    def scan_dataset_handedness(self) -> Dict[str, Tuple[Path, Optional[str]]]:
        table = {}
        for pid_dir in iter_numeric_id_dirs(self.data_dir):
            h = discover_handedness(pid_dir)
            table[pid_dir.name] = (pid_dir, h)
        return table

    def augment_dataset(self,
                        dry_run: bool = True,
                        target_participants: Optional[List[str]] = None,
                        max_new_participants: Optional[int] = None,
                        id_start: Optional[int] = None,
                        include_unknown: bool = False,
                        unknown_handedness_map: Optional[Dict[str, str]] = None,
                        unknown_default: str = "left") -> Dict[str, int]:
        print("🔍 Scanning dataset…")
        table = self.scan_dataset_handedness()
    
        left  = [(pid, pdir) for pid, (pdir, h) in table.items() if h == "left"]
        right = [(pid, pdir) for pid, (pdir, h) in table.items() if h == "right"]
        unknown = [(pid, pdir) for pid, (pdir, h) in table.items() if h not in ("left", "right")]
        print(f"Found  left={len(left)}, right={len(right)}, unknown={len(unknown)}")
    
        # IMPORTANT: include unknown if requested
        proc = left + right + (unknown if include_unknown else [])
    
        if target_participants:
            keep = set(target_participants)
            proc = [(pid, pdir) for pid, pdir in proc if pid in keep]
    
        if max_new_participants is not None:
            proc = proc[:max_new_participants]
    
        if not proc:
            print(" Nothing to augment.")
            return {"original_left":len(left),"original_right":len(right),
                    "unknown_handedness":len(unknown),"created_participants":0}
    
        nid = next_free_id(self.data_dir, id_start)
        print(f"First free new ID: id_{nid}")
    
        created = 0
        unknown_handedness_map = unknown_handedness_map or {}
    
        for pid, pdir in proc:
            measured = discover_handedness(pdir)
    
            # If unknown, force/assume handedness so we can flip deterministically
            if measured in ("left", "right"):
                handed = measured
            else:
                handed = unknown_handedness_map.get(pid, unknown_default)
                if handed not in ("left", "right"):
                    raise ValueError(f"Unknown handedness override for {pid} must be 'left' or 'right', got: {handed}")
    
            print(f"\n {pid} ({measured or 'unknown'} -> assumed {handed}) → id_{nid}")
            ok = self.create_mirrored_participant(pdir, nid, dry_run=dry_run)
            if ok:
                created += 1
                nid = next_free_id(self.data_dir, nid + 1)
    
        return {"original_left":len(left),"original_right":len(right),
                "unknown_handedness":len(unknown),"created_participants":created}


def run_hand_aug(data_dir: str,
                 dry_run: bool = True,
                 target_participants: Optional[List[str]] = None,
                 max_new_participants: Optional[int] = None,
                 id_start: Optional[int] = None,
                 seed: int = 42,
                 include_unknown: bool = False,
                 unknown_handedness_map: Optional[Dict[str, str]] = None,
                 unknown_default: str = "left"):
    data_dir = Path(data_dir).resolve()
    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory not found: {data_dir}")
    print(f"Using data directory: {data_dir}")
    eng = HandAugmentationEngine(data_dir, seed=seed)
    stats = eng.augment_dataset(
        dry_run=dry_run,
        target_participants=target_participants,
        max_new_participants=max_new_participants,
        id_start=id_start,
        include_unknown=include_unknown,
        unknown_handedness_map=unknown_handedness_map,
        unknown_default=unknown_default,
    )
    print("\n Hand Augmentation " + ("Simulation" if dry_run else "Complete") + "!")
    print(f"Original: left={stats['original_left']}, right={stats['original_right']}, unknown={stats['unknown_handedness']}")
    if not dry_run:
        print(f"Created mirrored participants: {stats['created_participants']}")
    return stats


# ---------- Example usage (uncomment to run) ----------
# DATASET_PATH = os.getenv("DATASET_PATH", "/workspace/base/src/data/eth_dataset")
# _ = run_hand_aug(DATASET_PATH, dry_run=True, max_new_participants=3)          # preview plan
# _ = run_hand_aug(DATASET_PATH, dry_run=False, max_new_participants=3, id_start=200)  # actually create a few


In [ ]:
import os, re
from pathlib import Path

# ---------------------------
# Resolve dataset root safely
# ---------------------------
def resolve_dataset_root() -> Path:
    # 1) Preferred: DATASET_PATH (base) + ETH_SUBPATH
    base = os.environ.get("DATASET_PATH") or os.environ.get("DATA_ROOT")
    sub  = os.environ.get("ETH_SUBPATH", "eth_dataset")

    candidates = []
    if base:
        candidates.append(Path(base) / sub)   # e.g. "data/eth_dataset"
        candidates.append(Path(base))         # in case user already points directly to eth_dataset

    # 2) Common absolute fallbacks (your list)
    candidates += [
        Path("/home/jupyterlab-docker/data/eth_dataset"),
        Path("/workspace/jupyterlab-docker/data/eth_dataset"),
        Path("data/eth_dataset"),
    ]

    # 3) If someone *did* set DATA_PATH, consider it too
    if os.environ.get("DATA_PATH"):
        candidates.insert(0, Path(os.environ["DATA_PATH"]))

    # pick first existing directory
    for p in candidates:
        if p.is_dir():
            return p.resolve()
a
    # diagnostics
    msg = ["Could not resolve dataset root. Tried:"]
    msg += [f"  - {str(p)} (exists={p.exists()})" for p in candidates]
    msg += [f"Env: DATASET_PATH={os.environ.get('DATASET_PATH')}, DATA_ROOT={os.environ.get('DATA_ROOT')}, ETH_SUBPATH={os.environ.get('ETH_SUBPATH')}, DATA_PATH={os.environ.get('DATA_PATH')}"]
    raise FileNotFoundError("\n".join(msg))

DATASET_ROOT = resolve_dataset_root()
print(" DATASET_ROOT =", DATASET_ROOT)

# ---------------------------
# Target IDs normalization
# ---------------------------
try:
    _target_source = TARGET_IDS
except NameError:
    _target_source = TRAIN

def _to_id(x):
    """Return 'id_<int>' from 'id_12', '12', 'ID-12', etc."""
    if isinstance(x, int):
        return f"id_{x}"
    s = str(x).strip()
    m = re.search(r"(\d+)", s)
    if not m:
        raise ValueError(f"Unrecognized participant id: {x}")
    return f"id_{int(m.group(1))}"

def _id_num(s):
    m = re.search(r"(\d+)", str(s))
    return int(m.group(1)) if m else 10**9

TARGET_IDS = [_to_id(x) for x in list(_target_source)]

# Sanity: keep only IDs that exist on disk
existing = {p.name for p in DATASET_ROOT.glob("id_*") if p.is_dir()}
target_existing = sorted([pid for pid in TARGET_IDS if pid in existing], key=_id_num)
missing = sorted([pid for pid in TARGET_IDS if pid not in existing], key=_id_num)

print(f"\nTarget TRAIN IDs requested: {len(TARGET_IDS)}")
print(f"Existing on disk: {len(target_existing)}")
if missing:
    print(f" Missing on disk (skipped): {missing[:20]}{' ...' if len(missing)>20 else ''}")

# target_existing is what you already computed
eng = HandAugmentationEngine(DATASET_ROOT, seed=42)
table = eng.scan_dataset_handedness()

unknown_in_train = [pid for pid in target_existing if table.get(pid, (None, None))[1] not in ("left", "right")]
print("Unknown handedness TRAIN IDs:", unknown_in_train)
print("Count:", len(unknown_in_train))



In [ ]:
# ---------------------------------------
# Force unknown handedness as LEFT and augment exactly 35 TRAIN participants
# ---------------------------------------
from pathlib import Path
import re

UNKNOWN_IN_TRAIN = ['id_23', 'id_25', 'id_33', 'id_34', 'id_35']
unknown_map = {pid: "left" for pid in UNKNOWN_IN_TRAIN}

ID_START = 51
N_NEW    = 35  # you want 35 mirrored participants

# --- Safety checks: ensure target ID range is free before writing ---
def _id_num(s):
    m = re.fullmatch(r"id_(\d+)", str(s))
    return int(m.group(1)) if m else None

existing_ids = {p.name for p in DATASET_ROOT.glob("id_*") if p.is_dir()}
planned_new  = [f"id_{i}" for i in range(ID_START, ID_START + N_NEW)]

already_there = [pid for pid in planned_new if pid in existing_ids]
if already_there:
    raise RuntimeError(
        "Refusing to write because these target IDs already exist:\n"
        f"{already_there}\n\n"
        "Fix: move/delete them OR choose a different ID_START."
    )

print("Planned new IDs are free:", planned_new[0], "…", planned_new[-1])

# --- DRY RUN (preview) ---
stats_dry = run_hand_aug(
    data_dir=DATASET_ROOT,
    dry_run=True,
    target_participants=target_existing,      # your 35 TRAIN ids that exist on disk
    max_new_participants=N_NEW,               # force exactly 35
    id_start=ID_START,                        # force id_51..id_85
    include_unknown=True,                     # include unknowns
    unknown_handedness_map=unknown_map,       # treat unknowns as LEFT
    unknown_default="left",
    seed=42
)

print("\n=== GENERATE (writes) ===")

# --- LIVE RUN (writes) ---
stats_live = run_hand_aug(
    data_dir=DATASET_ROOT,
    dry_run=False,
    target_participants=target_existing,
    max_new_participants=N_NEW,
    id_start=ID_START,
    include_unknown=True,
    unknown_handedness_map=unknown_map,
    unknown_default="left",
    seed=42
)

print("\nDone. Created:", stats_live.get("created_participants", "<?>"), "new mirrored participants.")


In [ ]:

from datetime import datetime
from threading import RLock
import csv
 
# Default configuration for processing
config = {
    'year': 2025, # The year in which the data was collected
    
    'default_watch_delta': 0, # Default delta if no manual value found
    'time_threshold': 45,  # the maximum tolerance for associating bin and feedback files (in seconds)
    
    'gesture_duration': 3, # Chosen duration of the gestures (in seconds)
    'tolerance_start': 1, # the maximum tolerance for finding peaks before the theoretical timestamp windows (in seconds)
    'tolerance_end': 2.5, # the maximum tolerance for finding peaks after the theoretical timestamp windows (in seconds)
    'fs': 100, # Sampling frequency (in Hertz)
}


# Default transform
centerscale = lambda x: (x - np.mean(x)) / (np.std(x) + 1e-8)

def align_ppg_to_acc(ppg_df, acc_df):
    # Ensure datetime
    ppg_df['UTCTime'] = pd.to_datetime(ppg_df['UTCTime'])
    acc_df['UTC'] = pd.to_datetime(acc_df['UTC'])

    # Set UTCTime as index
    ppg_interp = ppg_df.set_index('UTCTime')

    # Drop duplicate index entries (keep first)
    ppg_interp = ppg_interp[~ppg_interp.index.duplicated(keep='first')]

    # Select only PPG columns
    ppg_cols = [col for col in ppg_interp.columns if col.startswith('PPG')]
    ppg_interp = ppg_interp[ppg_cols]

    # Reindex on acc_df UTC, interpolate by time
    aligned_ppg = ppg_interp.reindex(acc_df['UTC']).interpolate(method='time')

    # Reset index to simple integer index
    aligned_ppg = aligned_ppg.reset_index(drop=True)

    return aligned_ppg


def load_csvs(acc_path, gyro_path, ppg_path):
    """Load everything from the CSV files and reformat the dataframe

    Args:
        acc_path (str): Full path to the accelerometer CSV file
        gyro_path (str): Full path to the gyroscope CSV file
        ppg_path (str): Full path to the PPG CSV file

    Returns:
        pd.DataFrame: A DataFrame containing the time series data with columns for accelerometer, gyroscope data and UTC time
    """
    acc_df = pd.read_csv(acc_path)
    gyro_df = pd.read_csv(gyro_path)
    ppg_df = pd.read_csv(ppg_path)
    
    acc_df = acc_df.rename(columns={"AccTime": "Timestamp"})
    
    gyro_df = gyro_df.rename(columns={"GyroTime": "Timestamp"})
    gyro_df = gyro_df.drop(columns=["UTC"]) 
    
    ppg_df = ppg_df.rename(columns={"PPGTime": "Timestamp"}) 
    aligned_ppg_df = align_ppg_to_acc(ppg_df, acc_df)  

    time_series_df = pd.concat([acc_df, gyro_df, aligned_ppg_df], axis=1)
    time_series_df.drop(columns=["Package_Num", "Timestamp"], inplace=True)
    time_series_df.rename(columns={
        col: col.replace('_100', '')
                .replace('_x', 'X')
                .replace('_y', 'Y')
                .replace('_z', 'Z')
        for col in time_series_df.columns 
    }, inplace=True) 
    time_series_df['UTC'] = pd.to_datetime(time_series_df['UTC'])
    return time_series_df


def nan_processing(time_series_df):
    """Handles NaN values

    Args:
        time_series_df (pd.DataFrame): DataFrame containing the time series data

    Returns:
        pd.DataFrame: DataFrame with NaN values handled (either replaced or dropped)
    """
    na_indices = time_series_df[time_series_df.isna().any(axis=1)].index.tolist()
    if len(na_indices) > 0:
        print(f"WARNING: There are {len(na_indices)} rows with NaN values. These will be replaced with 0.")
        print(f"Indices with NaN values: {na_indices}")
        print("Dropping rows with NaN values...")
        time_series_df = time_series_df[~time_series_df.isna().any(axis=1)].reset_index(drop=True)
    return time_series_df


def generate_timestamps(time_series_df, timestamps_preprocessed):
    """Generates timestamps dictionary using time references
    The timestamps dictionary is a dictionary with gesture names as keys and values
    is a dictionary with start time, end time, and the list of indices in the DataFrame
    at which the gesture occurs

    Args:
        time_series_df (pd.DataFrame): DataFrame containing the time series data with UTC timestamps
        timestamps_preprocessed (dict): Timestamps from the feedback file

    Returns:
        dict: timestamps dictionary
    """
    def parse_timestamp(timestamp_str):
        for fmt in ['%d/%m/%Y %H:%M:%S', '%m/%d/%Y, %I:%M:%S %p', '%d/%m/%Y, %H:%M:%S']:
            try:
                return datetime.strptime(timestamp_str, fmt)
            except ValueError:
                continue
        raise ValueError(f"Time data '{timestamp_str}' does not match known formats.")

    timestamps = {}
    for d in timestamps_preprocessed:
        ges = d["gesture"].lower().replace(" ", "_")
        t = parse_timestamp(d["gestureTimestamps"])
        if d["phase"] == "prepare":
            continue
        if ges not in timestamps and d["phase"] == "do":
            timestamps[ges] = {"start_time": t}
        else:
            timestamps[ges]["end_time"] = t

    for k, v in timestamps.items():
        time_series_df[f'mask_{k}'] = (time_series_df['UTC'] >= v['start_time']) & (time_series_df['UTC'] <= v['end_time'])
        indices = time_series_df[time_series_df[f'mask_{k}']].index
        v['indices'] = indices.tolist()

    return timestamps

def plot_and_save_magnitude(
    df: pd.DataFrame,
    timestamps: dict = None,
    windows: list = None,
    tolerance1: float = 1.0,
    tolerance2: float = 1.5,
    fs: int = 100,
    title: str = "Magnitude Plot",
    output_path: str = None, 
):
    """
    Plots and optionally saves the magnitude of motion with theoretical and detected gesture windows.

    Args:
        df (pd.DataFrame): Time series data containing 'AccX', 'AccY', 'AccZ', 'GyroX', 'GyroY', 'GyroZ'.
        timestamps (dict, optional): Theoretical timestamps dictionary.
        windows (list, optional): List of detected windows in the form [(gesture_name, start_idx, end_idx)].
        tolerance1 (float): Pre-gesture tolerance in seconds.
        tolerance2 (float): Post-gesture tolerance in seconds.
        fs (int): Sampling frequency.
        title (str): Title of the plot.
        output_path (str, optional): If provided, saves the figure to this path.
        show (bool): Whether to call plt.show().
    """
    plt.figure(figsize=(14, 5))

    # Compute magnitude of motion
    acc_mag = np.sqrt(df["AccX"]**2 + df["AccY"]**2 + df["AccZ"]**2)
    gyro_mag = np.sqrt(df["GyroX"]**2 + df["GyroY"]**2 + df["GyroZ"]**2)
    total_mag = acc_mag + gyro_mag

    plt.plot(total_mag.values, label="Motion Magnitude", color='black', linewidth=1)

    mean_val = np.mean(total_mag)
    plt.axhline(y=mean_val, color='blue', linestyle='--', label='Mean')

    label_drawn = {"theoretical": False, "tolerance": False, "detected": False}


    # Plot tolerance windows
    if timestamps and windows:
        for gesture_name, info in timestamps.items():
            if gesture_name in ["start_procedure", "end_procedure"]:
                continue
            if not info.get("indices"): continue
            start_idx, end_idx = info["indices"][0], info["indices"][-1]
            start_tol = max(0, int(start_idx - fs * tolerance1))
            end_tol = int(end_idx + fs * tolerance2)
            label = "Tolerance window" if not label_drawn["tolerance"] else None
            plt.axvspan(start_tol, end_tol, color='green', alpha=0.2, label=label)
            label_drawn["tolerance"] = True

    # Plot detected windows
    if windows: 
        for (gesture_name, start_idx, end_idx) in windows:
            label = "Detected window" if not label_drawn["detected"] else None
            plt.axvspan(start_idx, end_idx, color='orange', alpha=0.3, label=label)
            label_drawn["detected"] = True

    # Final plot formatting
    plt.title(title)
    plt.xlabel("Frame")
    plt.ylabel("Magnitude")
    plt.grid(True, linestyle='--', linewidth=0.5, alpha=0.7)
    plt.legend(loc="upper right")
    plt.tight_layout()

    # Save or show
    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        plt.savefig(output_path, dpi=300) 
        plt.close()
        
def plot_magnitude_and_adjust_timestamps(time_series_df, time_shift, gesture_windows, 
                   tolerance1=1, tolerance2=1.5, fs=100, 
                   plot_id="N/A", ax=None):
    """
    Plots motion magnitude over time, with original and shifted gesture windows.

    Args:
        time_series_df (pd.DataFrame): DataFrame with ['AccX', 'AccY', 'AccZ', 'GyroX', 'GyroY', 'GyroZ', 'timestamp'].
        time_shift (float): Time shift in seconds to apply to gesture windows.
        gesture_windows (dict): Dictionary of gestures, each containing start_time, end_time.
        tolerance1 (float): Seconds before gesture to mark.
        tolerance2 (float): Seconds after gesture to mark.
        fs (int): Sampling frequency in Hz.
        plot_id (str): ID for plot title.
        ax (matplotlib.axes.Axes): Optional axis to draw on.
    """

    if ax is None:
        fig, ax = plt.subplots(figsize=(15, 5))

    # Compute motion magnitude
    mag = np.sqrt(time_series_df["AccX"]**2 + time_series_df["AccY"]**2 + time_series_df["AccZ"]**2) + \
          np.sqrt(time_series_df["GyroX"]**2 + time_series_df["GyroY"]**2 + time_series_df["GyroZ"]**2)

    # Plot magnitude
    ax.plot(time_series_df['UTC'], mag, label="Motion Magnitude")
    ax.axhline(y=np.mean(mag), color='red', linestyle='--', label='Mean Magnitude')

    # Overlay gesture windows
    for i, (gesture_name, gesture_data) in enumerate(gesture_windows.items()): 
        if gesture_name == 'start_procedure' or gesture_name == 'end_procedure':
            continue
        start = gesture_data['start_time']
        end = gesture_data['end_time']
        shifted_start = start + pd.Timedelta(seconds=time_shift)
        shifted_end = end + pd.Timedelta(seconds=time_shift)

        # Plot original gesture window
        ax.axvspan(start - pd.Timedelta(seconds=tolerance1),
                   end + pd.Timedelta(seconds=tolerance2),
                   color='blue', alpha=0.2, label='original' if i == 0 else "")

        # Plot shifted gesture window
        ax.axvspan(shifted_start - pd.Timedelta(seconds=tolerance1),
                   shifted_end + pd.Timedelta(seconds=tolerance2),
                   color='green', alpha=0.3, label='shifted' if i == 0 else "")

    ax.set_title(f"Motion Magnitude Plot - {plot_id}")
    ax.set_xlabel("Time (UTC)")
    ax.set_ylabel("Magnitude")
    ax.legend(loc='upper right')
    ax.grid(True)

    # Only save/show if no external axis is passed
    if ax is None:
        if matplotlib.get_backend().lower() == 'agg':
            os.makedirs('tmp', exist_ok=True)
            plt.savefig(f'tmp/gesture_plot_{plot_id}.png')
        else:
            plt.show()
            
                
def bandpass_filter(signal, fs=100, lowcut=8.0, highcut=32.0, order=4):
    """Bandpass filtering

    Args:
        signal (pd.TimeSeries): Time series data to be filtered
        fs (int, optional): Sampling frequency. Defaults to 100.
        lowcut (float, optional): Lower frequency of the bandpass filter. Defaults to 8.0.
        highcut (float, optional): Higher frequency of the bandpass filter. Defaults to 32.0.
        order (int, optional): Order of the filter. Defaults to 4.

    Returns:
        np.array: Filtered signal
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)


def moving_average(x, w):
    """Smooths the signal using a moving average filter

    Args:
        x (np.array): motion signal
        w (int): window size for the moving average

    Returns:
        np.array: smoothed signal
    """
    return np.convolve(np.abs(x), np.ones(w)/w, mode='same')


def detect_gesture_windows(
    df: pd.DataFrame,
    timestamps: dict,
    fs: int = 100,
    gesture_duration: float = 1.5,
    tolerance_start: float = 0.5,
    tolerance_end: float = 1,
) -> List[Tuple[str, int, int]]:
    """Runs the peak detection algorithm

    Args:
        df (pd.DataFrame): DataFrame containing the time series data with accelerometer and gyroscope data
        timestamps (dict): Timestamps dictionary, as generated by generate_timestamps
        fs (int, optional): Sampling frequency. Defaults to 100.
        gesture_duration (float, optional): Desired duration in seconds for gesture windows. Defaults to 1.5.
        tolerance_start (float, optional): Tolerance for performing the gesture before the time reference. Defaults to 0.5.
        tolerance_end (float, optional): Tolerance for performing the gesture after the time reference. Defaults to 1.

    Returns:
        list: A list containing tuples of gesture name, start index, and end index for each detected gesture
    """
    # Step 1: Bandpass filter signals
    acc_x = bandpass_filter(df['AccX'], fs)
    acc_y = bandpass_filter(df['AccY'], fs)
    acc_z = bandpass_filter(df['AccZ'], fs)
    gyro_x = bandpass_filter(df['GyroX'], fs)
    gyro_y = bandpass_filter(df['GyroY'], fs)
    gyro_z = bandpass_filter(df['GyroZ'], fs)

    acc_filtered = np.sqrt(acc_x**2 + acc_y**2 + acc_z**2)
    gyro_filtered = np.sqrt(gyro_x**2 + gyro_y**2 + gyro_z**2)

    # Step 2: Sum magnitudes
    total_motion = acc_filtered + gyro_filtered

    # Step 3: Smooth with 1-sec moving average
    # smoothed = moving_average(total_motion, fs)
    smoothed = savgol_filter(total_motion, window_length=31, polyorder=3)

    # Step 4: Peak detection
    distance = fs  # 1 second between peaks
    # threshold = np.mean(smoothed)
    threshold = np.mean(smoothed) + 0.3 * np.std(smoothed)
    peaks, _ = find_peaks(smoothed, distance=distance, height=threshold)

    # Step 5: Filter using theoretical timestamps
    half_window = int((gesture_duration * fs) // 2)
    offset_start = int(tolerance_start * fs)
    offset_end = int(tolerance_end * fs)

    valid_windows = []
    for gesture, data in timestamps.items():
        if gesture == 'start_procedure' or gesture == 'end_procedure':
            continue
        # for peak in peaks:
        #     if data['indices'][0] - offset_start <= peak <= data['indices'][-1] + offset_end:
        #         start = max(0, peak - half_window)
        #         end = min(len(df), peak + half_window)
        #         if end - start == 2 * half_window:
        #             valid_windows.append((gesture, start, end))
        #         break  # only keep one peak per gesture

        candidate_peaks = []
        for peak in peaks:
            if data['indices'][0] - offset_start <= peak <= data['indices'][-1] + offset_end:
                candidate_peaks.append(peak)

        if candidate_peaks:
            # Find the peak with the maximum smoothed value
            best_peak = max(candidate_peaks, key=lambda p: smoothed[p])
            start = max(0, best_peak - half_window)
            end = min(len(df), best_peak + half_window)
            if end - start == 2 * half_window:
                valid_windows.append((gesture, start, end))
                
    return valid_windows


def plot_gestures(time_series_df, windows):
    """Plots the gestures detected

    Args:
        time_series_df (pd.DataFrame): DataFrame containing the time series data with accelerometer and gyroscope data
        windows (list): List of tuples containing gesture name, start index, and end index for each detected gesture
    """
    for i, (gesture, start, end) in enumerate(windows):
        plt.figure(figsize=(12, 4))
        plt.plot(centerscale(time_series_df["AccX"].values[start:end]), label="AccX")
        plt.plot(centerscale(time_series_df["AccY"].values[start:end]), label="AccY")
        plt.plot(centerscale(time_series_df["AccZ"].values[start:end]), label="AccZ")
        plt.plot(centerscale(time_series_df["GyroX"].values[start:end]), label="GyroX")
        plt.plot(centerscale(time_series_df["GyroY"].values[start:end]), label="GyroY")
        plt.plot(centerscale(time_series_df["GyroZ"].values[start:end]), label="GyroZ")
        plt.title(f"Gesture Window {i+1} ({gesture.replace('_', ' ').capitalize()})")
        plt.legend()
        if matplotlib.get_backend() == 'agg':
            plt.savefig('gesture_plot.png')
        else:
            plt.show()


def save_gestures(time_series_df, windows, folder):
    """Writes the gesture data organized by participant and gesture type.

    Args:
        time_series_df (pd.DataFrame): DataFrame with accelerometer and gyroscope data.
        windows (list): List of tuples: (gesture name, start index, end index).
        dataset_path (str): Root path to save data.
        feedback_file (str): Metadata info (filename source).
        participant_id (str): Unique identifier for participant (e.g., 'id_0').
    """
    for i, (gesture, start, end) in enumerate(windows): 
        file_path = os.path.join(folder, f"{gesture}.csv")

        with open(file_path, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(time_series_df.columns)
            for j in range(start, end):
                writer.writerow(time_series_df.iloc[j].values) 


def define_paths():
    load_dotenv()
    eth_path = os.path.join(os.getenv("DATASET_PATH"), os.getenv("ETH_SUBPATH"))
    
    bin_path = os.path.join(eth_path, "raw_data", "bin_files")
    feedback_path = os.path.join(eth_path, "raw_data", "feedback_files")
    watch_deltas_path = os.path.join(eth_path, "analysis", "watch_delta_shifts.json")

    if not os.path.exists(watch_deltas_path):
        raise Warning(f"Watch clock drift file not found.")
    
    if not os.path.exists(bin_path):
        raise FileNotFoundError(f"Folder {bin_path} does not exist.") 

    return bin_path, eth_path, feedback_path, watch_deltas_path


def rename_feedback_files(feedback_path):
    for file in os.listdir(feedback_path):
        if not file.endswith(".json"):
            continue

        full_path = os.path.join(feedback_path, file)
        with open(full_path, 'r') as f:
            data = json.load(f)

        gesture_data = data.get("gestureTimestamps")
        if not gesture_data:
            print(f"[SKIP] No gestureTimestamps in {file}")
            continue

        timestamp_str = gesture_data[0]["gestureTimestamps"]

        # Try multiple datetime formats
        parsed = False
        for fmt in ['%d/%m/%Y %H:%M:%S', '%m/%d/%Y, %I:%M:%S %p', '%Y-%m-%dT%H:%M:%S.%fZ']:
            try:
                timestamp = datetime.strptime(timestamp_str, fmt)
                parsed = True
                break
            except ValueError:
                continue

        if not parsed:
            print(f"[ERROR] Could not parse timestamp in {file}: {timestamp_str}")
            continue

        # Build new filename: feedback_YYYYMMDD_HHMMSS.json
        new_filename = f"feedback_{timestamp.strftime('%Y%m%d_%H%M%S')}.json"
        new_full_path = os.path.join(feedback_path, new_filename)

        # Avoid overwriting
        if os.path.exists(new_full_path):
            print(f"[SKIP] Target filename {new_filename} already exists.")
            continue

        os.rename(full_path, new_full_path)
        print(f"[RENAME] {file} -> {new_filename}")

def collect_timestamps(bin_path, feedback_path):
    bin_timestamps, feedback_timestamps = {}, {}

    for file in os.listdir(bin_path):
        if file.endswith(".bin"):
            timestamp = file.split('_')[-1].replace('.bin', '')
            dt = datetime.strptime(timestamp, '%m%d%H%M%S').replace(year=config['year'])
            bin_timestamps[dt] = file

    rename_feedback_files(feedback_path)

    for file in os.listdir(feedback_path):
        try:
            # Try reading with UTF-8 first, then with error handling for encoding issues
            try:
                with open(os.path.join(feedback_path, file), 'r', encoding='utf-8') as f:
                    data = json.load(f)
            except UnicodeDecodeError:
                # Try reading with different encoding or ignore errors
                with open(os.path.join(feedback_path, file), 'r', encoding='utf-8', errors='ignore') as f:
                    data = json.load(f)
            
            gesture_data = data.get("gestureTimestamps")
            if not gesture_data:
                print(f"[WARNING] Beginning timestamp not found in {file}, skipping...")
                continue
            
            timestamp_str = gesture_data[0]["gestureTimestamps"]

            # Try multiple datetime formats including the problematic ones
            parsed = False
            for fmt in ['%d/%m/%Y %H:%M:%S', '%d/%m/%Y, %H:%M:%S', '%m/%d/%Y, %I:%M:%S %p']:
                try:
                    timestamp = datetime.strptime(timestamp_str, fmt)
                    parsed = True
                    break
                except ValueError:
                    continue

            if not parsed:
                print(f"[ERROR] Could not parse timestamp in {file}: {timestamp_str}")
                continue

            feedback_timestamps[timestamp] = file
        except (json.JSONDecodeError, UnicodeDecodeError, Exception) as e:
            print(f"[ERROR] Could not process file {file}: {e}")
            continue

    return bin_timestamps, feedback_timestamps

def match_timestamps(bin_timestamps, feedback_timestamps):
    bin_keys = sorted(bin_timestamps)
    feedback_keys = sorted(feedback_timestamps)
    associated = []

    for i, bin_key in enumerate(bin_keys):
        matches = [
            j for j, feedback_key in enumerate(feedback_keys)
            if abs((bin_key - feedback_key).total_seconds()) < config['time_threshold']
        ]
        if len(matches) == 1:
            associated.append((i, matches[0]))
        elif len(matches) == 0:
            print(f"No feedback file found for bin file {bin_timestamps[bin_key]}")
        else:
            matched_files = [feedback_timestamps[feedback_keys[j]] for j in matches]
            print(f"Multiple feedback files found for bin file {bin_timestamps[bin_key]}: {matched_files}")

    print(f"Found {len(associated)} matches for {len(bin_keys)} bin files.")
    return bin_keys, feedback_keys, associated


def parse_data_if_needed(bin_path):

    print("Parsing data files...")
    online_files, offline_files = find_data_file(bin_path)
    
    for path, name in offline_files.items():
        offline_name = 'TEST_' + name
        if os.path.exists(os.path.join(bin_path, 'result', offline_name.strip(".bin"))):
            continue  
        parse_package_info(extract_offline_package(path), offline_name, bin_path)

    for path, name in online_files.items():
        parse_package_info(extract_package(path), name, bin_path)
        if is_exist_polarFile(bin_path, name):
            print("Moving polar file:", name)
            copy_polar_file(path, name)

    print("Finished parsing data files.")
    
parser_map = {
    "FFFF0000": "AccAndGyroParser",
    "FFFF0100": "AccAndGyroParser",
    "FFFF0500": "PPGParser",
    "FFFF0A00": "HRParser",
    # "FFFF0200": "MAGParser",
    # "FFFF0400": "BaroParser",
    "FFFF0700": "ECGParser",
    "FFFF0B00": "SPo2Parser",
    "FFFF0C00": "TemperParser",
    "FFFF0D00": "BatteryTempParser",
    "FFFF0E00": "ScreenTempParser",
    # "FFFF0F00": "GPSParser",
    "FFFF1000": "SettingParser",
    "FFFF1100": "PointParser",
    "FFFF1200": "TPICParser",
}

class SingletonType(type):
    single_lock = RLock()
    instance_map = dict()

    def __call__(cls, *args, **kwargs):
        with SingletonType.single_lock:
            if SingletonType.instance_map.get(args[1]) is None:
                SingletonType.instance_map[args[1]] = super(SingletonType, cls).__call__(*args, **kwargs)

        return cls.instance_map.get(args[1])

    @classmethod
    def clear(cls):
        cls.instance_map.clear()


def convert_to_int(little_byte_st, signed=True):
    return int.from_bytes(bytearray.fromhex(little_byte_st), "little", signed=signed)


# 小端存储‘<f’,大端存储'>f'
def convert_to_float(little_byte_st):
    return struct.unpack('<f', bytes.fromhex(little_byte_st))[0]


def covert_to_utc(utc_time):
    _time = datetime.fromtimestamp(int(utc_time / 1000))
    return _time.strftime("%Y-%m-%d %H:%M:%S")


class AccAndGyroParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "ACC_" if sensor_flag == "FFFF0000" else "Gyro_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'AccTime', 'UTC', 'Acc_x_100', 'Acc_y_100', 'Acc_z_100']) if sensor_flag == "FFFF0000" else self._csv_write.writerow(['Package_Num', 'GyroTime', 'UTC', 'Gyro_x_100', 'Gyro_y_100', 'Gyro_z_100'])

    def do_parser(self, raw_data_line: str) -> str:
        # print(raw_data_line)
        if len(raw_data_line) == 716:  # 兼容固定字节版本
            self.do_single_package_parser(raw_data_line)
        else:  # 动态长度 + 组包
            tmp_str = raw_data_line
            while tmp_str.startswith(self.sensor_flag):
                data_len = convert_to_int(tmp_str[8:12], False)
                # "FFFF0000" + 4(长度) + 16(time) + 16 (utc) + acclen + 8(package_num) + 4 ("FFFF")
                offset = 8 + 4 + 16 + 16 + data_len * 2 + 8 + 4
                self.do_single_package_parser(tmp_str[:offset])
                tmp_str = tmp_str[offset:]

    def do_single_package_parser(self, single_line):
        data_len = convert_to_int(single_line[8:12], False)
        data_time = convert_to_int(single_line[12:28], False)
        utc = covert_to_utc(convert_to_int(single_line[28:44], False))
        #utc = convert_to_int(single_line[28:44], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        offset = 44
        for i in range(int(data_len / 6)):
            x = convert_to_int(single_line[offset:offset + 4], True)
            offset = offset + 4
            y = convert_to_int(single_line[offset:offset + 4], True)
            offset = offset + 4
            z = convert_to_int(single_line[offset:offset + 4], True)
            offset = offset + 4
            self._csv_write.writerow([package_num, data_time, utc, x, y, z])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


class PPGParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "PPG_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        # self._csv_write.writerow(['Package_Num', 'PPGTime', 'UTCTime'] + [p + '_100' for p in ['PPG' + str(i+1) for i in range(20)]] + ['Press_cnt'])
        self._csv_write.writerow(['Package_Num', 'PPGTime', 'UTCTime'] + [p + '_100' for p in ['PPG' + str(i+1) for i in range(20)]])

    def do_parser(self, single_line):
        data_len = convert_to_int(single_line[8:12], False)
        data_time = convert_to_int(single_line[12:28], False)
        utc = covert_to_utc(convert_to_int(single_line[28:44], False))
        #utc = convert_to_int(single_line[28:44], False)
        channel_num = convert_to_int(single_line[44:48], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        row_num = int(data_len / (channel_num * 4))
        press_cnt = convert_to_int(single_line[-16:-12])  # 新增的press_cnt计数
        offset = 48
        for row in range(int(row_num)):
            ppg_data = []
            for channel in range(channel_num):
                ppg_data.append(convert_to_int(single_line[offset:offset + 8], True))
                offset = offset + 8
            # self._csv_write.writerow([package_num, data_time, utc] + ppg_data + [press_cnt])
            self._csv_write.writerow([package_num, data_time, utc] + ppg_data)

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


class HRParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "HR_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'Time', 'UTC', "bpm", "status", "fitCurrentSpeed", "fitCadence", "fitAltitude", "fitBpm"])

    def do_parser(self, single_line):
        data_time = convert_to_int(single_line[12:28], False)
        utc = covert_to_utc(convert_to_int(single_line[28:44], False))
        #utc = convert_to_int(single_line[28:44], False)
        bpm = convert_to_int(single_line[44:52], False)
        status = convert_to_int(single_line[52:60], False)
        fitCurrentSpeed = convert_to_float(single_line[60:68])
        fitCadence = convert_to_int(single_line[68:76], False)
        fitAltitude = convert_to_int(single_line[76:84], False)
        fitBpm = convert_to_int(single_line[84:92], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        self._csv_write.writerow([package_num, data_time, utc, bpm, status, fitCurrentSpeed, fitCadence, fitAltitude, fitBpm])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


class PointParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "Point_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'TPtime', 'UTC', 'noNdeNum'])

    def do_parser(self, single_line):
        data_len = convert_to_int(single_line[8:12], False)
        utc = covert_to_utc_polar(convert_to_int(single_line[28:44], False))
        package_num = convert_to_int(single_line[-12:-4], False)
        if data_len != 1:
            EndTimpStamp = convert_to_int(single_line[44:60], False)
            offset = 60
            for row in range(int(data_len)):
                tp_data = convert_to_int(single_line[offset:offset + 2], False)
                offset = offset + 2
                self._csv_write.writerow([int(package_num),  EndTimpStamp, utc, tp_data])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()

class TPICParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "TPIC_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'TPIC', 'xChannel', 'yChannel', 'threshold'])

    def do_parser(self, single_line):
        data_len = convert_to_int(single_line[8:12], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        if data_len == 1:
            ICtp = convert_to_int(single_line[44:46], False)
            xChannel = convert_to_int(single_line[46:48], False)
            yChannel = convert_to_int(single_line[48:50], False)
            threshold = convert_to_int(single_line[50:52], False)
            self._csv_write.writerow([package_num, ICtp, xChannel, yChannel, threshold])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()

class MAGParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "Mag_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'MagTime', 'UtcTime', "axisX", "axisY", "axisZ", "Accuracy", "offsetX", "offsetY", "offsetZ"])

    def do_parser(self, single_line):
        data_time = convert_to_int(single_line[12:28], False)
        UtcTime = covert_to_utc(convert_to_int(single_line[28:44], False))
        offsetX = convert_to_float(single_line[44:52])
        offsetY = convert_to_float(single_line[52:60])
        offsetZ = convert_to_float(single_line[60:68])
        axisX = convert_to_float(single_line[76:84])
        axisY = convert_to_float(single_line[84:92])
        axisZ = convert_to_float(single_line[92:100])

        Accuracy = convert_to_int(single_line[68:76], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        self._csv_write.writerow([package_num, data_time, UtcTime, axisX, axisY, axisZ, Accuracy, offsetX, offsetY, offsetZ])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


class BaroParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "Baro_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'BaroTime', "UTCTime", "pressure", "temperature", "dataValid"])

    def do_parser(self, single_line):
        data_time = convert_to_int(single_line[12:28], False)
        utc = covert_to_utc(convert_to_int(single_line[28:44], False))
        #utc = convert_to_int(single_line[28:44], False)
        pressure = convert_to_int(single_line[44:52], False)
        temperature = convert_to_int(single_line[52:60], False)
        dataValid = convert_to_int(single_line[60:68], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        self._csv_write.writerow([package_num, data_time, utc, pressure, temperature, dataValid])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


class ECGParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "ECG_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'ECGTime', "data"])

    def do_parser(self, single_line):
        data_len = convert_to_int(single_line[8:12], False)
        data_time = convert_to_int(single_line[12:28], False)
        UtcTime = covert_to_utc(convert_to_int(single_line[28:44], False))
        #ECGData = convert_to_int(single_line[44:44+8*50], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        offset = 44
        ecg_data = []
        for i in range(int(data_len / 4)):
            ecg_data.append(convert_to_float(single_line[offset:offset + 8]))
            offset = offset + 8
        self._csv_write.writerow([package_num, data_time] + ecg_data)

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


class SPo2Parser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "SPO2_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'HRTime', "UtcTime", "Bmd", "altitude", "Status"])

    def do_parser(self, single_line):
        data_time = convert_to_int(single_line[12:28], False)
        UtcTime = covert_to_utc(convert_to_int(single_line[28:44], False))
        Bmd = convert_to_int(single_line[44:48], False)
        altitude = convert_to_int(single_line[48:52], False)
        Status = convert_to_int(single_line[52:56], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        self._csv_write.writerow([package_num, data_time, UtcTime, Bmd, altitude, Status])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


class TemperParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "Temper_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Packge_Num', 'endTime', 'UTCTime', 'TimeStamp', 'ENVTempera', 'ShellTempera',
                                   'LEDTempera',
                                   'BatteryTempera',
                                   'sport_status', 'BodyTempera', 'SkinTempera', 'AmbientTempera', 'TempConfidence',
                                   'TempScenario'])

    def do_parser(self, single_line):
        data_time = covert_to_utc(convert_to_int(single_line[12:28], False))
        UtcTime = convert_to_int(single_line[28:44], False)
        timeStamp = convert_to_int(single_line[44:52], False)
        eNVTempera = convert_to_int(single_line[52:60], False)
        shellTempera = convert_to_int(single_line[60:68], False)
        lEDTempera = convert_to_int(single_line[68:76], False)
        batteryTempera = convert_to_int(single_line[76:84], False)
        sport_status = convert_to_int(single_line[84:88], False)
        bodyTempera = convert_to_int(single_line[88:92], False)
        skinTempera = convert_to_int(single_line[92:96], False)
        ambientTempera = convert_to_int(single_line[96:100], False)
        tempConfidence = convert_to_int(single_line[100:104], False)
        tempScenario = convert_to_int(single_line[104:108], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        self._csv_write.writerow([package_num, data_time, UtcTime, timeStamp, eNVTempera/100, shellTempera/100,
                                  lEDTempera/10, batteryTempera/100, sport_status, bodyTempera/10, skinTempera/10,
                                  ambientTempera, tempConfidence, tempScenario])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


class BatteryTempParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "BatteryTemp_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'BatteryTime', "temperature"])

    def do_parser(self, single_line):
        data_time = convert_to_int(single_line[12:28], False)
        temperature = convert_to_int(single_line[44:48], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        self._csv_write.writerow([package_num, data_time, temperature])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


class ScreenTempParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "ScreenTemp_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'ScreenTime', "temperature"])

    def do_parser(self, single_line):
        data_time = convert_to_int(single_line[12:28], False)
        temperature = convert_to_int(single_line[44:48], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        self._csv_write.writerow([package_num, data_time, temperature])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


class GPSParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "GPS_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'Endtime', "UTCTime"])

    def do_parser(self, single_line):
        #data_time = convert_to_int(single_line[12:28], False)
        #utc = covert_to_utc(convert_to_int(single_line[28:44], False))
        temperature = convert_to_int(single_line[44:48], False)
        package_num = convert_to_int(single_line[-12:-4], False)
        self._csv_write.writerow([package_num, data_time, utc])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


class SettingParser(metaclass=SingletonType):
    def __init__(self, result_file_home, sensor_flag):
        self.sensor_flag = sensor_flag
        csv_title_prefix = "Setting_"
        csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
        self._result_path = os.path.join(result_file_home, csv_file_name)
        self._f_io = open(self._result_path, 'a', newline='', encoding='UTF8')
        self._csv_write = csv.writer(self._f_io, dialect='excel')
        self._csv_write.writerow(['Package_Num', 'Endtime', "UTCTime", "PhaseNum", "Flag", "rfgain", "Current", "dcoffset", "ParamType", "period", "option", "kkvalue"])

    def do_parser(self, single_line):
        data_time = convert_to_int(single_line[12:28], False)
        UtcTime = covert_to_utc(convert_to_int(single_line[28:44], False))
        ParamType = convert_to_int(single_line[44:52], False)
        period = convert_to_int(single_line[52:60], False)
        option = convert_to_int(single_line[60:68], False)
        offset = 68 + 8
        for i in range(0, 20):
            PhaseNum = convert_to_int(single_line[offset:offset + 2], True)
            offset = offset + 2
            Flag = convert_to_int(single_line[offset:offset + 2], True)
            offset = offset + 2
            rfgain = convert_to_int(single_line[offset:offset + 4])
            offset = offset + 4
            Current = convert_to_int(single_line[offset:offset + 8], True)
            offset = offset + 8
            dcoffset = convert_to_int(single_line[offset:offset + 8], True)
            offset = offset + 8
            # phaseCoef老产品参数现不用
            #phaseCoef = convert_to_float(single_line[offset:offset + 8])
            #offset = offset + 8
            kkvalue = convert_to_float(single_line[-20:-12])
            package_num = convert_to_int(single_line[-12:-4], False)
            self._csv_write.writerow([package_num, data_time, UtcTime, PhaseNum, Flag, rfgain, Current, dcoffset, ParamType, period, option, kkvalue])

    def close_io(self):
        if self._f_io is not None:
            self._f_io.close()


def find_data_file(path):
    online_file_paths = {}
    offline_file_paths = {}
    for root, dirs, fs in os.walk(path):
        for i in fs:
            filePath = os.path.join(root, i)
            if filePath.find('.txt') != -1 and filePath.find('meta') == -1 and filePath.find(
                    'desc') == -1 and filePath.find('result') == -1 and filePath.find('package') == -1\
                    and filePath.find('data_collect_') == -1:
                online_file_paths[filePath] = i

    for root, dirs, fs in os.walk(path):
        offline_file_paths.update(dict([(os.path.join(root, file_name), file_name) for file_name in fs if file_name.endswith("bin")]))

    print(online_file_paths)
    print(offline_file_paths)
    return online_file_paths, offline_file_paths


def extract_frame_header_info(frame_head_str):
    frame_len = int(frame_head_str[2:6], 16)
    frame_control = int(frame_head_str[6:8], 16)
    frame_fsn = 0 if frame_control == 0 else int(frame_head_str[8:10], 16)
    return frame_len, frame_control, frame_fsn


def extract_package(raw_file_path):
    with open(raw_file_path) as fp:
        lines = fp.readlines()
        raw_data_str = lines[0]
    offset = 0
    package_list = []
    package_data = ""
    while offset < len(raw_data_str):
        if raw_data_str[offset: offset + 2] != '5A':
            raise Exception("frame start tag exception")
        (frame_len, frame_control, frame_fsn) = extract_frame_header_info(raw_data_str[offset: offset + 10])
        frame_header_str_len = 8 if frame_control == 0 else 10
        pay_load_len = frame_len - 1 if frame_control == 0 else frame_len - 2
        pay_load_str_offset = offset + frame_header_str_len
        pay_load = raw_data_str[pay_load_str_offset: pay_load_str_offset + pay_load_len * 2]
        offset = offset + frame_header_str_len + pay_load_len * 2 + 4  # 2字节CRC
        package_data = package_data + pay_load
        if frame_control == 0 or frame_control == 3:
            package_list.append(package_data)
            package_data = ""

    package_file = raw_file_path.replace(".txt", "_package.txt")
    with open(package_file, "w+") as fp:
        fp.writelines("\n".join(package_list))

    return package_list


def extract_offline_package(raw_file_path):
    hex_data = ""
    with open(raw_file_path, 'rb') as f:
        readLen = 0
        while True:
            raw_data = f.read(20000)
            readLen += raw_data.__len__()
            hex_data += "".join([hex(byte_data)[-2:].upper() for byte_data in raw_data]).replace('X', '0')
            if raw_data.__len__() < 20000:
                break

    # path = raw_file_path.replace(".bin", ".txt")
    sample_datas = ['FFFF' + data + 'FFFF' for data in hex_data[4:].split('FECAFECA')]
    # with open(path, "w") as hex_file:
    #     hex_file.write('\n'.join(sample_datas))
    return sample_datas


def is_frame_2_byte(frame_content):
    len_flag = convert_to_int(frame_content[6:8], False)
    return len_flag > 127


def parse_package_info(package_list, raw_file_name, PATH):
    result_file_path = os.path.join(PATH, "result", raw_file_name[:-4])
    if not os.path.exists(result_file_path):
        os.makedirs(result_file_path)

    parser = None
    for package_info in package_list:
        if not package_info.startswith("0A09") and not package_info.startswith("FFFF"):
            print("Invalid package: " + package_info)
            continue
        if package_info.startswith("FFFF"):
            raw_begin_offset = 0
        else:
            raw_begin_offset = 18 if is_frame_2_byte(package_info) else 16
        raw_begin_content = package_info[raw_begin_offset:raw_begin_offset+8]
        if parser_map.get(raw_begin_content) is None:
            # print("Invalid Sensor Type: " + raw_begin_content)
            continue

        if raw_begin_content in ['FFFF0F00', 'FFFF0400'] :
            continue
        parser = eval(parser_map.get(raw_begin_content))(result_file_path, raw_begin_content)
        try:
            parser.do_parser(package_info[raw_begin_offset:])
        except:
            print('file name:', raw_file_name, 'error')
            continue

    SingletonType.clear()


def copy_polar_file(polar_filepath, polar_filename):
    polar_filename = polar_filename.replace(".txt", ".csv")
    polar_filepath = polar_filepath.replace(".txt", ".csv")
    result_file_path = os.path.join(PATH, "result", polar_filename[:-4])
    csv_title_prefix = "Polar_"
    csv_file_name = csv_title_prefix + time.strftime('%Y%m%d%H%M%S', time.localtime(time.time())) + '.csv'
    src_file = polar_filepath
    dst_file = os.path.join(result_file_path, csv_file_name)

    shutil.copyfile(src_file, dst_file)

    df = pd.read_csv(dst_file, header=None, names=['PolarTime', 'Value'])
    df.to_csv(dst_file, index=False)


def is_exist_polarFile(path, polar_filename):
    polar_filename = polar_filename.replace(".txt", ".csv")
    for root, dirs, fs in os.walk(path):
        for i in fs:
            filePath = os.path.join(root, i)
            if filePath.find(polar_filename) != -1:
                return True

    return False




In [ ]:
# %% Walking augmentation (robust): augment every non-walking folder
import os, re, math, random, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

# ---------------- CONFIG ----------------
SEED           = 42
WIN_LEN        = 300
WIN_STRIDE     = 300
SCALE_ACC      = (0.4, 0.9)
SCALE_GYRO     = (0.2, 0.6)
KEEP_PPG       = True
COPY_FEEDBACK  = True
VERBOSE        = True
DRY_RUN        = False   # set True to preview

random.seed(SEED)
np.random.seed(SEED)

# ---------------- Resolve paths ----------------
def resolve_dataset_root() -> Path:
    base = os.environ.get("DATASET_PATH") or os.environ.get("DATA_ROOT")
    sub  = os.environ.get("ETH_SUBPATH", "eth_dataset")

    candidates = []
    if base:
        candidates += [Path(base) / sub, Path(base)]
    candidates += [
        Path("/home/jupyterlab-docker/data/eth_dataset"),
        Path("/workspace/jupyterlab-docker/data/eth_dataset"),
        Path("data/eth_dataset"),
    ]
    for p in candidates:
        if p.is_dir():
            return p.resolve()
    raise FileNotFoundError("Could not resolve DATASET_ROOT. Tried:\n" + "\n".join(map(str, candidates)))

DATASET_ROOT = resolve_dataset_root()
print(" DATASET_ROOT =", DATASET_ROOT)

def resolve_walking_csv(dataset_root: Path, fname="walking_youssef.csv") -> Path:
    # you said: "same location as the folder eth_dataset" => parent of DATASET_ROOT
    candidates = [dataset_root / fname, dataset_root.parent / fname]
    for p in candidates:
        if p.is_file():
            return p.resolve()
    raise FileNotFoundError(f"walking CSV not found. Tried: {candidates}")

WALKING_CSV = resolve_walking_csv(DATASET_ROOT)
print(" WALKING_CSV =", WALKING_CSV)

# ---------------- Targets ----------------
# Use your intended 35 participants:
# TARGET_IDS = TRAIN   # if TRAIN exists in your notebook
# or hardcode / load them.

try:
    #TARGET_IDS = list(TRAIN)  # your 35
    TARGET_IDS = list(NEW_MIRRORED)
except NameError:
    raise NameError("Define TRAIN (your 35 ids) first, then re-run this cell.")

# Keep only those that exist
TARGET_IDS = sorted([pid for pid in TARGET_IDS if (DATASET_ROOT / pid).is_dir()],
                    key=lambda s: int(re.search(r"(\d+)", s).group(1)))
print(f"Target IDs: {len(TARGET_IDS)}")

# ---------------- Column helpers ----------------
def _acc_cols(df):
    cx = "AccX" if "AccX" in df.columns else ("Acc_X" if "Acc_X" in df.columns else None)
    cy = "AccY" if "AccY" in df.columns else ("Acc_Y" if "Acc_Y" in df.columns else None)
    cz = "AccZ" if "AccZ" in df.columns else ("Acc_Z" if "Acc_Z" in df.columns else None)
    return (cx, cy, cz)

def _gyro_cols(df):
    cx = "GyroX" if "GyroX" in df.columns else ("Gyro_X" if "Gyro_X" in df.columns else None)
    cy = "GyroY" if "GyroY" in df.columns else ("Gyro_Y" if "Gyro_Y" in df.columns else None)
    cz = "GyroZ" if "GyroZ" in df.columns else ("Gyro_Z" if "Gyro_Z" in df.columns else None)
    return (cx, cy, cz)

def _has_csvs(d: Path) -> bool:
    return d.is_dir() and any(d.glob("*.csv"))

def _load_walking_chunks(csv_path: Path, win=300, stride=300):
    dfw = pd.read_csv(csv_path)

    ax, ay, az = _acc_cols(dfw)
    gx, gy, gz = _gyro_cols(dfw)
    if None in (ax, ay, az, gx, gy, gz):
        raise ValueError("walking CSV must contain AccX/Y/Z (or Acc_X/Y/Z) and GyroX/Y/Z (or Gyro_X/Y/Z).")

    Ax, Ay, Az = (dfw[c].to_numpy() for c in (ax, ay, az))
    Gx, Gy, Gz = (dfw[c].to_numpy() for c in (gx, gy, gz))
    L = min(map(len, [Ax,Ay,Az,Gx,Gy,Gz]))
    Ax, Ay, Az, Gx, Gy, Gz = [a[:L] for a in (Ax,Ay,Az,Gx,Gy,Gz)]

    chunks = []
    for s in range(0, L - win + 1, stride):
        chunks.append({
            "Acc":  np.stack([Ax[s:s+win], Ay[s:s+win], Az[s:s+win]], axis=1),  # [win,3]
            "Gyro": np.stack([Gx[s:s+win], Gy[s:s+win], Gz[s:s+win]], axis=1),
        })
    if VERBOSE:
        print(f" Walking chunks: {len(chunks)} windows of {win} (stride {stride})")
    if not chunks:
        raise ValueError("No walking chunks created (walking CSV too short for WIN_LEN).")
    return chunks

def _choose_overlay(chunks, pid=None, fname=None):
    """
    Works with either call:
      - _choose_overlay(chunks)
      - _choose_overlay(chunks, pid, fname)
    Deterministic when pid+fname provided.
    """
    if pid is None or fname is None:
        ch = random.choice(chunks)
        a_scale = random.uniform(*SCALE_ACC)
        g_scale = random.uniform(*SCALE_GYRO)
        return ch["Acc"] * a_scale, ch["Gyro"] * g_scale

    key = f"{pid}::{fname}".encode("utf-8")
    h = int(hashlib.md5(key).hexdigest(), 16)
    rng = random.Random(h)
    ch = chunks[h % len(chunks)]
    a_scale = rng.uniform(*SCALE_ACC)
    g_scale = rng.uniform(*SCALE_GYRO)
    return ch["Acc"] * a_scale, ch["Gyro"] * g_scale

def _overlay_into(df: pd.DataFrame, acc_chunk: np.ndarray, gyro_chunk: np.ndarray):
    T = len(df); W = len(acc_chunk)
    reps = math.ceil(T / W)
    acc_ol = np.tile(acc_chunk, (reps, 1))[:T]
    gyr_ol = np.tile(gyro_chunk, (reps, 1))[:T]

    ax, ay, az = _acc_cols(df)
    gx, gy, gz = _gyro_cols(df)

    if None not in (ax, ay, az):
        arr = df[[ax, ay, az]].to_numpy()
        df[[ax, ay, az]] = arr + acc_ol

    if None not in (gx, gy, gz):
        arr = df[[gx, gy, gz]].to_numpy()
        df[[gx, gy, gz]] = arr + gyr_ol

    return df

def walking_target_name(src_folder: str) -> str:
    s = src_folder.strip()
    if s == "standing":
        return "walking"
    if s == "sitting":
        return "walking_sitting"
    return f"walking_{s}"

# ---------------- Run ----------------
chunks = _load_walking_chunks(WALKING_CSV, WIN_LEN, WIN_STRIDE)

participants_done = 0
folders_created = 0
files_written = 0

for pid in TARGET_IDS:
    pdir = DATASET_ROOT / pid

    # all condition folders that do NOT contain "walking"
    cond_folders = [
        d for d in pdir.iterdir()
        if d.is_dir()
        and ("walking" not in d.name.lower())
        and _has_csvs(d)
    ]

    if not cond_folders:
        if VERBOSE: print(f" {pid}: no non-walking folders with CSVs → skip")
        continue

    if VERBOSE:
        print(f"\n {pid}: {len(cond_folders)} source folders → augment")

    participants_done += 1

    for src_dir in sorted(cond_folders, key=lambda x: x.name):
        tgt_name = walking_target_name(src_dir.name)
        tgt_dir  = pdir / tgt_name

        # already exists with csvs
        if _has_csvs(tgt_dir):
            if VERBOSE: print(f"✓ {pid}: {tgt_name}/ exists → skip")
            continue

        if VERBOSE:
            print(f"👟 {pid}: creating {tgt_name}/ from {src_dir.name} ({len(list(src_dir.glob('*.csv')))} files)")

        if not DRY_RUN:
            tgt_dir.mkdir(parents=True, exist_ok=True)
            folders_created += 1

            if COPY_FEEDBACK:
                fb = src_dir / "feedback.json"
                if fb.exists():
                    (tgt_dir / "feedback.json").write_text(fb.read_text())

        wrote_here = 0
        for f in sorted(src_dir.glob("*.csv")):
            try:
                if DRY_RUN:
                    wrote_here += 1
                    continue

                df = pd.read_csv(f)
                acc_chunk, gyr_chunk = _choose_overlay(chunks, pid=pid, fname=f"{src_dir.name}/{f.name}")
                df_aug = _overlay_into(df.copy(), acc_chunk, gyr_chunk)
                df_aug.to_csv(tgt_dir / f.name, index=False)

                wrote_here += 1
                files_written += 1
            except Exception as e:
                print(f"  ✗ {pid}/{src_dir.name}/{f.name}: {e}")

        # cleanup empty target folder (avoid leaving junk)
        if (not DRY_RUN) and wrote_here == 0:
            try:
                # remove feedback.json if it exists, then rmdir
                fb_out = tgt_dir / "feedback.json"
                if fb_out.exists():
                    fb_out.unlink()
                tgt_dir.rmdir()
                folders_created -= 1
            except Exception:
                pass

        if VERBOSE and not DRY_RUN:
            print(f"  ✓ wrote {wrote_here} CSVs → {tgt_dir}")

print(f"\n Done. participants={participants_done}, folders_created={folders_created}, files_written={files_written}")
